## Notebook de Práctica (resuelto)

Actividades optativas para comprender y practicar con modelos de redes neuronales, tanto MLP como CNN, para el procesamiento de imágenes.

Trabajamos con el dataset https://www.kaggle.com/datasets/prasunroy/natural-images: 6899 imágenes divididas en 8 clases (airplane, car, cat, dog, flower, fruit, motorbike, person).

In [ ]:
!pip install keras torch tensorflow matplotlib scipy Pillow kagglehub

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

### 0. Descarga del dataset

Usamos `kagglehub` para descargarlo y dejamos un enlace `./natural_images` apuntando a la carpeta que contiene las 8 clases, para que el resto del código funcione con rutas relativas.

In [ ]:
import kagglehub
from pathlib import Path

download_path = Path(kagglehub.dataset_download("prasunroy/natural-images"))

# Buscar la carpeta que contiene directamente las clases (ej. 'airplane')
data_dir = next(p.parent for p in download_path.rglob("airplane") if p.is_dir())
print("Carpeta de clases:", data_dir)

if not Path("./natural_images").exists():
    os.symlink(data_dir, "./natural_images")

print(sorted(p.name for p in Path("./natural_images").iterdir()))

### 1. Exploración del dataset

1. Cargamos **diez imágenes aleatorias** con `keras.utils.load_img`.
2. Por cada imagen mostramos sus dimensiones y luego intentamos crear un arreglo con `np.array`.
3. Mostramos las diez imágenes en una grilla de cinco filas y dos columnas, con su etiqueta.

In [ ]:
import random
from pathlib import Path

random.seed(42)

def random_files_from_directories(
        dirs: list[str], *,
        file_count: int = 10,
        base: str = "./"
    ) -> list[tuple[str, str]]:
    """Retorna una lista de pares (directorio, archivo) elegidos al azar."""
    base_path = Path(base)
    selected_directories = random.choices(sorted(dirs), k=file_count)
    return [
        (directory, str(random.choice(sorted((base_path / directory).iterdir()))))
        for directory in selected_directories
    ]

class_directories = sorted(path.name for path in Path("./natural_images").iterdir())
files = random_files_from_directories(class_directories, base="./natural_images/")
files

In [ ]:
import keras
import numpy as np

images = []
for label, path in files:
    img = keras.utils.load_img(path)          # objeto PIL
    arr = keras.utils.img_to_array(img)       # (alto, ancho, 3)
    images.append((label, arr))
    print(f"{label:10s} tamaño PIL (ancho x alto) = {img.size}  -> array shape = {arr.shape}")

**¿Tienen todas las mismas dimensiones?** No. Cada imagen del dataset tiene su propio tamaño (por ejemplo, las de autos y personas suelen tener proporciones distintas). Solo coincide la cantidad de canales (3, RGB).

In [ ]:
# Intento de crear un arreglo con imágenes de distintas dimensiones
try:
    batch = np.array([arr for _, arr in images])
    print("Shape:", batch.shape)
except ValueError as e:
    print("Error:", e)

**Respuesta**: Al tener dimensiones diferentes, `np.array` no puede armar un tensor rectangular y lanza un `ValueError` (*inhomogeneous shape*). Esto nos dice que **un tensor necesita que todos sus elementos tengan la misma forma**: para entrenar una red con lotes de imágenes primero hay que redimensionarlas a un tamaño común. Además, las capas densas de un MLP tienen un número fijo de entradas, así que también lo exigen.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 2, figsize=(8, 16))
for ax, (label, arr) in zip(axes.flat, images):
    ax.imshow(arr.astype("uint8"))
    ax.set_title(f"{label} {arr.shape[:2]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

4. Cargamos el dataset con `keras.utils.image_dataset_from_directory`, dividiéndolo en entrenamiento (80%) y validación (20%), con `image_size` de 64x64.

In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "./natural_images",
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

In [ ]:
# Recorremos cada conjunto como indica la consigna
for name, dataset in [("train", train_ds), ("val", val_ds)]:
    n = 0
    for images_b, labels_b in dataset:
        n += images_b.shape[0]
    print(f"{name}: {n} imágenes, shape de un lote: {tuple(images_b.shape)}")

In [ ]:
# Primeras 10 imágenes del conjunto de entrenamiento en una grilla
for images_b, labels_b in train_ds.take(1):
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for i, ax in enumerate(axes.flat):
        img = images_b[i].numpy().astype("uint8")
        ax.imshow(img)
        ax.set_title(f"{class_names[int(labels_b[i])]}")
        ax.axis("off")
        print(f"Imagen {i}: shape={img.shape}, etiqueta={int(labels_b[i])} ({class_names[int(labels_b[i])]})")
    plt.tight_layout()
    plt.show()

**Diferencias con la grilla del punto 3**: ahora **todas las imágenes tienen el mismo shape `(64, 64, 3)`**, porque `image_dataset_from_directory` las redimensiona. Como resultado se ven pixeladas (menos resolución) y algunas **deformadas**, ya que se estiran o comprimen para llegar a 64x64 sin respetar su proporción original. Además, las etiquetas ya no son strings sino enteros (índice de la carpeta en orden alfabético).

**`crop_to_aspect_ratio` y `pad_to_aspect_ratio`**: ambos evitan la distorsión manteniendo la relación de aspecto.
- `crop_to_aspect_ratio=True`: recorta el centro de la imagen para que tenga la proporción del `image_size` (cuadrada en este caso) y después redimensiona. Se pierde algo del borde.
- `pad_to_aspect_ratio=True`: agrega bandas de relleno (negras) a los costados o arriba/abajo hasta lograr la proporción, sin perder contenido.

Solo se puede usar uno de los dos a la vez.

In [ ]:
# Comparación visual de las tres estrategias con una misma imagen
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, kwargs, title in zip(
        axes,
        [{}, {"crop_to_aspect_ratio": True}, {"pad_to_aspect_ratio": True}],
        ["Por defecto (deforma)", "crop_to_aspect_ratio", "pad_to_aspect_ratio"]):
    ds = keras.utils.image_dataset_from_directory(
        "./natural_images", image_size=IMG_SIZE, batch_size=1,
        shuffle=False, verbose=False, **kwargs)
    for img, _ in ds.skip(1500).take(1):
        ax.imshow(img[0].numpy().astype("uint8"))
    ax.set_title(title)
    ax.axis("off")
plt.show()

### 2. Creación y Configuración de modelos.

5. **MLP con al menos dos capas densas.**

**¿Es necesaria alguna tarea de preprocesamiento?** Sí:
- **Normalizar** los píxeles de [0, 255] a [0, 1] con una capa `Rescaling(1/255)`, para que el entrenamiento sea estable.
- **Aplanar** cada imagen `(64, 64, 3)` en un vector de 12.288 valores con `Flatten`, porque las capas densas reciben vectores.

Incluimos ambas como capas del modelo, así el mismo preprocesamiento se aplica automáticamente al predecir.

In [ ]:
from keras import layers

mlp = keras.Sequential([
    keras.Input(shape=(64, 64, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
], name="mlp")

6. **Función de pérdida**: usamos **Sparse Categorical Cross Entropy**, porque:
- Es un problema de **clasificación multiclase** (8 clases mutuamente excluyentes) con salida softmax; la entropía cruzada mide qué tan lejos está la distribución predicha de la clase correcta.
- Las etiquetas que entrega `image_dataset_from_directory` (con `label_mode="int"` por defecto) son **enteros**, no vectores one-hot. Si fueran one-hot usaríamos `categorical_crossentropy`.

7. **Compilación**: optimizador Adam.

**¿Cuál es la tarea del optimizador?** Ajustar los pesos de la red para minimizar la función de pérdida: con los gradientes calculados por backpropagation decide cuánto y en qué dirección mover cada peso. **Sí, se puede usar otro** (SGD, RMSprop, AdamW, etc.); el optimizador **no depende del tipo de tarea** (clasificación o regresión), eso lo determina la función de pérdida. Lo que cambia entre optimizadores es la velocidad y estabilidad de la convergencia. Adam es una buena opción por defecto porque adapta el learning rate de cada parámetro.

In [ ]:
mlp.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

8. **Red neuronal convolucional (CNN)**:
- a) Dos capas `Conv2D` (aprenden filtros que detectan bordes, texturas y formas).
- b) Dos capas `MaxPooling2D` (reducen la resolución a la mitad, conservando los rasgos más fuertes).
- c) Una capa fully-connected (`Dense`) para la clasificación final, más la salida softmax.

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(64, 64, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),                 # 64x64 -> 32x32
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),                 # 32x32 -> 16x16
    layers.Flatten(),
    layers.Dense(128, activation="relu"),        # fully-connected
    layers.Dense(num_classes, activation="softmax"),
], name="cnn")

9. Compilamos la CNN con la misma función de pérdida y optimizador que el MLP.

In [ ]:
cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

### 3. Entrenamiento

10. Entrenamos ambos modelos durante 10 épocas usando el conjunto de validación, comparamos parámetros con `.summary()` y medimos el tiempo de entrenamiento.

11. Graficamos la pérdida de entrenamiento y validación con el objeto `History`.

In [ ]:
mlp.summary()
cnn.summary()
print(f"Parámetros MLP: {mlp.count_params():,}")
print(f"Parámetros CNN: {cnn.count_params():,}")

In [ ]:
import time

EPOCHS = 10

t0 = time.time()
history_mlp = mlp.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
time_mlp = time.time() - t0

t0 = time.time()
history_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
time_cnn = time.time() - t0

print(f"Tiempo MLP: {time_mlp:.1f} s")
print(f"Tiempo CNN: {time_cnn:.1f} s")

**Comparación de parámetros y velocidad**: el MLP tiene muchos **más parámetros** (~3,2 millones) que la CNN (~2,1 millones), porque su primera capa densa conecta cada uno de los 12.288 píxeles con cada una de sus 256 neuronas. En la CNN, los filtros convolucionales son pequeños (3x3) y **comparten pesos** a lo largo de toda la imagen, así que tienen muy pocos parámetros; la mayoría de los parámetros de la CNN están en la capa densa después del `Flatten`.

Sin embargo, el MLP suele **entrenar más rápido** por época: aunque tiene más parámetros, una multiplicación de matrices densa es una operación barata. La convolución, en cambio, aplica cada filtro en todas las posiciones de la imagen (64x64 y luego 32x32), lo que implica muchas más operaciones por imagen, especialmente sin GPU. Es decir, **la cantidad de parámetros no es lo mismo que la cantidad de cómputo**.

*(Confirmar con los tiempos impresos arriba; si se usa GPU la diferencia puede achicarse.)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, h) in zip(axes, [("MLP", history_mlp), ("CNN", history_cnn)]):
    epochs = range(1, len(h.history["loss"]) + 1)
    ax.plot(epochs, h.history["loss"], marker="o", label="Pérdida entrenamiento")
    ax.plot(epochs, h.history["val_loss"], marker="o", label="Pérdida validación")
    ax.set_title(name)
    ax.set_xlabel("Época")
    ax.set_ylabel("Pérdida")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación**: en ambos modelos la pérdida de entrenamiento baja época tras época. Si la pérdida de validación se estanca o empieza a subir mientras la de entrenamiento sigue bajando, el modelo está **sobreajustando** (memoriza el entrenamiento). Esto suele notarse antes en la CNN, que aprende rápido; se podría mitigar con `Dropout` o *data augmentation*.

### 4. Evaluación

12. Evaluamos ambos modelos sobre el conjunto de validación.

In [ ]:
mlp_loss, mlp_acc = mlp.evaluate(val_ds, verbose=0)
cnn_loss, cnn_acc = cnn.evaluate(val_ds, verbose=0)
print(f"MLP -> pérdida: {mlp_loss:.4f} | accuracy: {mlp_acc:.4f}")
print(f"CNN -> pérdida: {cnn_loss:.4f} | accuracy: {cnn_acc:.4f}")

**Comparación**: la CNN suele obtener **mayor exactitud y menor pérdida** que el MLP (típicamente más de 85-90% contra 60-75%). La CNN aprovecha la estructura espacial de la imagen (píxeles vecinos relacionados) y detecta patrones sin importar dónde aparecen; el MLP, al aplanar, pierde esa información y trata cada píxel como una variable independiente.

13. Tomamos una imagen del conjunto de validación, la mostramos y la predecimos con ambos modelos.

**Pistas aplicadas**: los modelos esperan lotes con shape `(N, 64, 64, 3)`, así que a una imagen `(64, 64, 3)` le agregamos la dimensión del lote con `np.expand_dims(..., axis=0)`. No dividimos por 255 porque la capa `Rescaling` ya está dentro de los modelos.

In [ ]:
TARGET_CLASS = "car"   # clase elegida; la imagen externa del punto 14 debe ser de esta misma clase
target_idx = class_names.index(TARGET_CLASS)

val_img, val_label = None, None
for images_b, labels_b in val_ds:
    for img, lab in zip(images_b, labels_b):
        if int(lab) == target_idx:
            val_img, val_label = img.numpy(), int(lab)   # arreglo NumPy (64, 64, 3)
            break
    if val_img is not None:
        break

print("Shape de la imagen:", val_img.shape, "| clase real:", class_names[val_label])

plt.imshow(val_img.astype("uint8"))
plt.title(f"Clase real: {class_names[val_label]}")
plt.axis("off")
plt.show()

In [ ]:
def predict_one(model, img):
    """Recibe una imagen (alto, ancho, canales) y devuelve (clase, probabilidad)."""
    batch = np.expand_dims(img, axis=0)          # (1, 64, 64, 3)
    probs = model.predict(batch, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx])

print("Input shape del modelo:", mlp.input_shape, "| shape del lote:", np.expand_dims(val_img, 0).shape)
for name, model in [("MLP", mlp), ("CNN", cnn)]:
    cls, p = predict_one(model, val_img)
    print(f"{name}: predice '{cls}' con {p*100:.1f}% de probabilidad")

14. **Imagen externa**: descargamos de Internet una imagen de la **misma clase** (`car`), con un solo objeto visible sobre un fondo sencillo, y la subimos al entorno (en Colab: panel *Archivos* → subir, o la celda de abajo). Le aplicamos el mismo preprocesamiento que en el entrenamiento: redimensionar a 64x64 (la normalización la hace la capa `Rescaling`).

In [ ]:
# En Colab, descomentar para subir la imagen desde la computadora:
# from google.colab import files
# uploaded = files.upload()
# EXTERNAL_PATH = next(iter(uploaded))

EXTERNAL_PATH = "imagen_externa.jpg"   # reemplazar por el nombre del archivo descargado

ext_img = keras.utils.img_to_array(keras.utils.load_img(EXTERNAL_PATH, target_size=IMG_SIZE))
ext_batch = np.expand_dims(ext_img, axis=0)

plt.imshow(ext_img.astype("uint8"))
plt.title("Imagen externa")
plt.axis("off")
plt.show()

print("Shape del lote preparado:", ext_batch.shape)
for name, model in [("MLP", mlp), ("CNN", cnn)]:
    cls, p = predict_one(model, ext_img)
    print(f"{name}: predice '{cls}' con {p*100:.1f}% de probabilidad")

**Comparación**: con la imagen externa es común que el **MLP se equivoque o baje mucho su confianza**, porque aprendió patrones atados a posiciones exactas de píxeles y a la distribución de colores del dataset. La CNN suele mantener la predicción correcta porque reconoce formas (ruedas, ventanas) independientemente de su posición exacta. Aun así, una imagen de una fuente distinta (otra iluminación, fondo o encuadre) puede bajar la confianza de ambos.

15. **Robustez ante un desplazamiento de la imagen.**

**Expectativas antes de aplicar la transformación:**
- **¿Esperamos que cambien las predicciones?** Un desplazamiento de 3-4 píxeles es casi imperceptible para una persona, así que idealmente no deberían cambiar; pero esperamos que al menos cambien las probabilidades.
- **¿Esperamos que el MLP y la CNN reaccionen igual?** No. El MLP asocia cada peso a una posición fija de píxel: al mover la imagen, cada neurona recibe valores distintos y la salida puede cambiar bastante. La CNN es aproximadamente **invariante a traslaciones** gracias a que sus filtros recorren toda la imagen (pesos compartidos) y al *max pooling*, así que esperamos que sea más estable.
- **¿Mismo comportamiento en la imagen de validación y en la externa?** Probablemente no: en la imagen externa los modelos ya tienen menos confianza, así que un pequeño cambio tiene más chances de cambiar la clase predicha.

In [ ]:
from scipy.ndimage import shift
import numpy as np

# Las funciones reciben una imagen sin la dimensión del lote: (alto, ancho, canales).

def translate_image(
    img: np.ndarray,
    pixels_x: int,
    pixels_y: int,
    fill_with = 0.0
) -> np.ndarray:
  """
  Desplaza una imagen una cantidad de `pixels_x` a la derecha y
  `pixels_y` hacia abajo.

  Rellena los píxeles faltantes con `fill_with` (por defecto, negro)

  :returns La imagen desplazada
  """
  shifted = shift(
      img,
      shift=(pixels_y, pixels_x, 0),
      mode='constant',
      cval=fill_with
  )
  return shifted

In [ ]:
SHIFT_X, SHIFT_Y = 4, 3

def compare_shift(name, img):
    shifted = translate_image(img, SHIFT_X, SHIFT_Y)

    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    axes[0].imshow(img.astype("uint8")); axes[0].set_title(f"{name}: original"); axes[0].axis("off")
    axes[1].imshow(shifted.astype("uint8")); axes[1].set_title(f"{name}: desplazada"); axes[1].axis("off")
    plt.show()

    for model_name, model in [("MLP", mlp), ("CNN", cnn)]:
        cls_before, p_before = predict_one(model, img)
        cls_after, p_after = predict_one(model, shifted)
        changed = "SÍ cambió" if cls_before != cls_after else "no cambió"
        print(f"  {model_name}: antes '{cls_before}' ({p_before*100:.1f}%) -> "
              f"después '{cls_after}' ({p_after*100:.1f}%) | clase {changed} | "
              f"Δprob = {(p_after - p_before)*100:+.1f} pts")

compare_shift("Validación", val_img)
compare_shift("Externa", ext_img)

**Conclusiones** *(ajustar según los valores obtenidos al ejecutar)*:
- **¿Qué modelo mantuvo su predicción de forma más estable?** La **CNN**: su clase predicha se mantiene y la probabilidad varía poco, gracias a los filtros compartidos y al max pooling. En el MLP la probabilidad cambia más y en algunos casos cambia incluso la clase.
- **¿Diferencias entre la imagen de validación y la externa?** Sí: con la imagen de validación (misma distribución que el entrenamiento) ambos modelos parten de una confianza alta y el desplazamiento afecta poco; con la imagen externa la confianza inicial es menor, por lo que el desplazamiento tiene más impacto, sobre todo en el MLP.
- **¿Coincidieron con las expectativas?** En general sí: el experimento muestra la principal ventaja de las CNN para imágenes, que es reconocer un patrón aunque cambie de posición. La CNN no es perfectamente invariante (el pooling solo tolera desplazamientos pequeños), por eso también cambia un poco su probabilidad.